# Exploratory Factor Analysis (EFA)

1. Descriptive Statistics
2. Standardized Values (Z-Scores)
3. Pearson Correlation Matrix
4. Exploratory Factor Analysis (EFA)
   1. Determining the Number of Factors
   2. Maximum Likelihood Estimation
   3. Total Variance Explained
   4. Component Matrix
   5. Unrotated Factor Matrix
   6. Rotated Factor Matrix
   7. Communalities
   8. Factor Scores
   9. Factor Scores by Brand

In [ ]:
#  Imports
!pip install factor_analyzer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import LinearSegmentedColormap
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

pd.set_option("display.max_columns", None)

In [ ]:
# Load Dataset
df = pd.read_csv("")

## 1. Descriptive Statistics

First, descriptive statistics are calculated for all nine brand perception items.

In [ ]:
# Select items
items = ["perform", "leader", "latest", "fun", "serious",
         "bargain", "value", "trendy", "rebuy"]

# Calculate descriptive statistics for the selected items
descriptive_stats = df[items].describe().T[
    ["mean", "std", "min", "max"] # Select the mean, standard deviation, minimum and maximum
]

# Output
print("\nDescriptive Statistics:")
print("-----------------------------")
print(descriptive_stats.round(3))

## 2. Standardized Values (Z-Scores)

The brand perception variables are standardized using z-scores.

In [ ]:
# Calculate the z-scores for each item by subtracting its mean and dividing by its standard deviation
z_scores = (
        (df[items] - df[items].mean()) / df[items].std()
).add_prefix("Z_")

# Group the z-scores by brand
means_z_by_brand = z_scores.groupby(df["brand"]).mean().round(3)

# Output
print("\nMean standardized values by brand:")
print("-----------------------------")
print(means_z_by_brand)

## 3. Pearson Correlation Matrix

A Pearson correlation matrix is calculated for the nine brand perception items.

In [ ]:
# Calculate the correlation matrix
R = df[items].corr(method="pearson")

# Output
print("\nPearson Correlation Matrix:")
print("-----------------------------")
print(R.round(3).to_string())

- Look at the strongest positive correlations (e.g.> 0.5). What groups of items can you identify?
- Look at the near-zero correlations. What items move independently from another?
- Look at the strongest negative correlations (e.g.< -0.3). Which items are distinct from each other?

In [ ]:
# Coloured correlation matrix

# Create a custom blue-white-blue colormap
blue_white_blue = LinearSegmentedColormap.from_list(
    "blue_white_blue",
    ["#08306B", "#FFFFFF", "#08306B"]
)

# Create a small figure with a high resolution
plt.figure(figsize=(5, 4), dpi=150)

# Create the correlation heatmap
sns.heatmap(
    R, # Use the correlation matrix as the data source
    annot=True, # Display the correlation values inside each cell
    fmt=".3f", # Display correlation values with three decimal places
    annot_kws={"size": 6},  # Set the font size of the annotations
    cmap=blue_white_blue, # Apply the custom blue-white-blue color map
    vmin=-1, # Set the minimum value of the color scale
    vmax=1, # Set the maximum value of the color scale
    center=0, # Center the color scale at zero
    square=True, # Make all heatmap cells square
    linewidths=0.3,  # Set the thickness of the grid lines
    cbar_kws={"shrink": 0.5}  # Reduce the size of the color bar
)

# Set the font size of the axis labels
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)

# Adjust the layout to fit all elements
plt.tight_layout()

# Output
plt.show()

# 4. Exploratory Factor Analysis (EFA)

An Exploratory Factor Analysis is conducted to identify latent dimensions underlying the nine brand perception items.

In [ ]:
# Suitability for Factor Analysis

# Perform Bartlett's test of sphericity
chi_square, bartlett_p = calculate_bartlett_sphericity(df[items])

# Calculate the Kaiser-Meyer-Olkin (KMO)
kmo_per_item, kmo_total = calculate_kmo(df[items])

# Output
print(f"Bartlett's test of sphericity: chi-square = {chi_square:.3f}, "
      f"{'p < .001' if bartlett_p < 0.001 else f'p = {bartlett_p:.3f}'}")

print(f"KMO (overall): {kmo_total:.3f}")

print("\nKMO per item:")
print(pd.Series(kmo_per_item, index=items, name="KMO").round(3).to_string())

## 4.1 Determine the Number of Factors

The eigenvalues are the sum of squared factor loadings (based on a Principal Component Analysis).
- Factor with eigenvalue of 1 explains 1/9 (we have 9 items) of total variance
- Hence rule that factor/component should at least explain as much variance (Kaiser criterion)

In [ ]:
# Eigenvalues > 1

# Calculate the eigenvalues
eigenvalues = np.linalg.eigvalsh(R.to_numpy())[::-1]

# Calculate the percentage of total variance explained by each factor
variance_percent = eigenvalues / eigenvalues.sum() * 100

# Calculate the cumulative percentage of variance explained by the factors
cumulative_percent = np.cumsum(variance_percent)

# Create a DataFrame containing the eigenvalues and explained variance
eigenvalues_df = pd.DataFrame({
    "Factor": range(1, len(eigenvalues) + 1), # Assign a number to each factor
    "Eigenvalue": eigenvalues, # Store the eigenvalue of each factor
    "% of Variance": variance_percent, # Store the percentage of variance explained by each factor
    "Cumulative %": cumulative_percent # Store the cumulative percentage of explained variance
}).round(3)  # Round all numerical values to three decimal places

# Output
print(eigenvalues_df.to_string(index=False))

# Determine the number of factors with eigenvalues greater than or equal to 1 according to the Kaiser criterion
n_factors = int((eigenvalues >= 1).sum())

# Output
print(f"\nNumber of factors according to Kaiser criterion: {n_factors}")

In [ ]:
# Scree plot - Elbow

# Create a small figure for the scree plot
plt.figure(figsize=(4.5, 3.2), dpi=150)

# Plot the eigenvalues against their corresponding factor numbers
plt.plot(
    eigenvalues_df["Factor"],  # Set factor numbers on the x-axis
    eigenvalues_df["Eigenvalue"],  # Set eigenvalues on the y-axis
    marker="o",  # Display a marker for each eigenvalue
    linewidth=1.5  # Set the thickness of the line
)

# Add the eigenvalue as a label above each data point
for x, y in zip(eigenvalues_df["Factor"], eigenvalues_df["Eigenvalue"]):
    plt.annotate(
        f"{y:.3f}",  # Format the eigenvalue to three decimal places
        (x, y),  # Set the position of the annotation
        textcoords="offset points",  # Position the label relative to the data point
        xytext=(0, 6),  # Offset the label by 6 points vertically
        ha="center",  # Center the label horizontally
        fontsize=6  # Reduce the annotation font size
    )

# Add a horizontal reference line at an eigenvalue of 1
plt.axhline(
    y=1,
    linestyle="--",  # Use a dashed line
    linewidth=0.8  # Set the reference line thickness
)

# Add axis labels and a title
plt.xlabel("Factor Number", fontsize=8)
plt.ylabel("Eigenvalue", fontsize=8)
plt.title("Scree Plot", fontsize=10, pad=6)

# Display a tick for each factor number
plt.xticks(eigenvalues_df["Factor"], fontsize=7)
plt.yticks(fontsize=7)

# Add a subtle dashed grid
plt.grid(True, alpha=0.2, linestyle="--")

# Adjust the layout to prevent overlapping elements
plt.tight_layout()

# Output
plt.show()

In [ ]:
# Chi-square test - insignificance

# Select the relevant items and remove rows with missing values
data = df[items].dropna()

# Test factor models with different numbers of factors
for k in range(1, len(items) + 1):

    # Initialize a maximum likelihood factor analysis model without rotation
    fa = FactorAnalyzer(n_factors=k, method="ml", rotation=None)

    # Fit the factor analysis model to the data
    fa.fit(data)

    # Perform the factor sufficiency test to assess model fit
    chi2, df_test, p = fa.sufficiency(len(data))

    # Print the chi-square statistic, degrees of freedom and p-value
    print(f"{k} factors: χ² = {chi2:.2f}, df = {df_test}, p = {p:.3f}")

## 4.2 Maximum Likelihood Estimation

Maximum Likelihood estimation identifies factor loadings by maximizing the likelihood of observing the data under the assumed factor model.

In [ ]:
# Initialize an unrotated factor analysis model using maximum likelihood estimation
fa_unrotated = FactorAnalyzer(
    n_factors=n_factors,  # Set the number of factors to extract
    method="ml",  # Use maximum likelihood estimation
    rotation=None  # Do not apply factor rotation
)

# Fit the unrotated factor analysis model to the selected items
fa_unrotated.fit(df[items])

# Apply Varimax rotation if more than one factor is extracted
rotation = "varimax" if n_factors > 1 else None

# Initialize the factor analysis model with the selected rotation method
fa = FactorAnalyzer(
    n_factors=n_factors,  # Set the number of factors to extract
    method="ml",  # Use maximum likelihood estimation
    rotation=rotation  # Apply the selected rotation method
)

# Fit the factor analysis model to the selected items
fa.fit(df[items])

## 4.3 Total Variance Explained

The total variance explained indicates the proportion of the dataset's overall variance accounted for by the retained components.

In [ ]:
# Extract the sums of squared loadings, proportion of variance, and cumulative proportion for the unrotated factor solution
ext_ss, ext_prop, ext_cum = fa_unrotated.get_factor_variance()

# Extract the corresponding variance statistics for the rotated factor solution
rot_ss, rot_prop, rot_cum = fa.get_factor_variance()

# Create a DataFrame summarizing the variance explained before and after rotation
total_variance_explained = pd.DataFrame({
    "Factor": range(1, n_factors + 1),  # Assign a number to each factor
    "Initial Eigenvalue": eigenvalues[:n_factors],  # Store the initial eigenvalues
    "Extraction SS Loadings": ext_ss,  # Store the sums of squared loadings after factor extraction
    "Extraction %": ext_prop * 100,  # Calculate the percentage of variance explained after extraction
    "Extraction Cum. %": ext_cum * 100,  # Store the cumulative percentage of variance explained after extraction
    "Rotation SS Loadings": rot_ss,  # Store the sums of squared loadings after rotation
    "Rotation %": rot_prop * 100,  # Calculate the percentage of variance explained after rotation
    "Rotation Cum. %": rot_cum * 100,  # Store the cumulative percentage of variance explained after rotation
}).round(3)  # Round all numerical values to three decimal places

# Output
print("\nTotal Variance Explained:")
print("-----------------------------")
print(total_variance_explained.to_string(index=False))

## 4.4 Component Matrix

PCA identifies the underlying structure of the data and reduces dimensionality by extracting components based on the Kaiser criterion (eigenvalues ≥ 1).

In [ ]:
# Initialize a principal component analysis model without rotation
pca = FactorAnalyzer(
    n_factors=n_factors,  # Set the number of components to extract
    method="principal",  # Use the principal factor extraction method
    rotation=None  # Do not apply component rotation
)

# Fit the model to the selected items
pca.fit(df[items])

# Create names for the extracted components
component_names = [f"Component {i + 1}" for i in range(n_factors)]

# Create a DataFrame containing the component loadings
component_matrix = pd.DataFrame(
    pca.loadings_,  # Retrieve the loading values for each item
    index=items,  # Use the item names as row labels
    columns=component_names  # Use the component names as column labels
).round(3)  # Round all loading values to three decimal places

# Output
print("\nComponent Matrix:")
print("-----------------------------")
print(component_matrix.to_string())

## 4.5 (Unrotated) Factor Matrix

The urotated factor matrix shows the initial loadings of each item on the factors, where the first factor captures as much variance as possible, which often makes the pattern hard to interpret because many items load on several factors.

In [ ]:
# Create names for the extracted factors
factor_names = [f"Factor {i + 1}" for i in range(n_factors)]

# Create a DataFrame containing the unrotated factor loadings
factor_matrix = pd.DataFrame(
    fa_unrotated.loadings_,  # Retrieve the unrotated loading values
    index=items,  # Use the item names as row labels
    columns=factor_names  # Use the factor names as column labels
).round(3)  # Round all loading values to three decimal places

# Output
print("\nFactor Matrix (Unrotated):")
print("-----------------------------")
print(factor_matrix.to_string())

In [ ]:
# Create a small figure for the factor loading heatmap
plt.figure(figsize=(4.5, 3.5), dpi=150)

# Create a heatmap of the unrotated factor loadings
sns.heatmap(
    factor_matrix,  # Use the factor loading matrix as the data source
    annot=True,  # Display factor loading values inside each cell
    fmt=".3f",  # Display values with three decimal places
    annot_kws={"size": 6},  # Set the font size of the annotations
    cmap=blue_white_blue,  # Apply the custom blue-white-blue color map
    vmin=-1,  # Set the minimum value of the color scale
    vmax=1,  # Set the maximum value of the color scale
    center=0,  # Center the color scale at zero
    square=True,  # Make all heatmap cells square
    linewidths=0.3,  # Set the thickness of the grid lines
    cbar_kws={"shrink": 0.5}  # Reduce the size of the color bar
)

# Set the font size of the axis labels
plt.xticks(fontsize=7);
plt.yticks(fontsize=7);

# Output
plt.show()

## 4.6 Rotated Factor Matrix

The rotated factor matrix shows the loadings after rotation (e.g. varimax), which redistributes the variance so that each item loads strongly on one factor and weakly on the others, making the factors easier to interpret.

In [ ]:
# Create a DataFrame containing the rotated factor loadings
rotated_factor_matrix = pd.DataFrame(
    fa.loadings_,  # Retrieve the rotated factor loading values
    index=items,  # Use the item names as row labels
    columns=factor_names  # Use the factor names as column labels
).round(3)  # Round all loading values to three decimal places

# Output
print("\nRotated Factor Matrix:")
print("-----------------------------")
print(rotated_factor_matrix.to_string())

In [ ]:
# Create a small figure for the rotated factor loading heatmap
plt.figure(figsize=(4.5, 3.5), dpi=150)

# Create a heatmap of the rotated factor loadings
sns.heatmap(
    rotated_factor_matrix,  # Use the rotated factor loading matrix as the data source
    annot=True,  # Display factor loading values inside each cell
    fmt=".3f",  # Display values with three decimal places
    annot_kws={"size": 6},  # Set the font size of the annotations
    cmap=blue_white_blue,  # Apply the custom blue-white-blue color map
    vmin=-1,  # Set the minimum value of the color scale
    vmax=1,  # Set the maximum value of the color scale
    center=0,  # Center the color scale at zero
    square=True,  # Make all heatmap cells square
    linewidths=0.3,  # Set the thickness of the grid lines
    cbar_kws={"shrink": 0.5}  # Reduce the size of the color bar
)

# Set the font size of the axis labels
plt.xticks(fontsize=7);
plt.yticks(fontsize=7);

# Output
plt.show()

## 4.7 Communalities

- Communalities are the sum of squared factor loadings across factors, per item
- The "Initial" column represents the starting estimate of how much variance in each variable is shared with the other variables, before any factors are actually extracted
- Interpretation like 𝑅2 for the regression $$
\text{item} = \sum_i \text{factor}_i
$$

In [ ]:
# Estimate the initial communalities using squared multiple correlations (SMC)
smc = 1 - 1 / np.diag(np.linalg.inv(R.to_numpy()))

# Create a DataFrame comparing the initial communalities estimated using SMC with the communalities extracted by the factor analysis model
communalities = pd.DataFrame({
    "Initial (SMC)": smc,  # Store the initial communalities estimated using squared multiple correlations
    "Extraction": fa.get_communalities()  # Retrieve the communalities after factor extraction
}, index=items).round(3)  # Round all values to three decimal places

# Output
print("\nCommunalities:")
print("-----------------------------")
print(communalities.to_string())

## 4.8 Factor Scores

- Factor scores are a computed values for each factor per observation for subsequent analysis → Use 3 factor scores instead of 9 items
- Simple and intuitive approaches would be (weighted) sum of items with loadings above certain cutoff, e.g.> 0,60

In [ ]:
# Calculate factor scores for each observation and store them in a DataFrame
factor_scores_df = pd.DataFrame(
    fa.transform(df[items]),  # Transform the selected items into factor scores
    columns=factor_names,  # Use the factor names as column labels
    index=df.index  # Preserve the original DataFrame index
).round(3)  # Round all factor scores to three decimal places

# Output
print("\nFactor Scores:")
print("-----------------------------")
print(factor_scores_df.head().to_string())

## 4.9 Factor Scores by Brand

In [ ]:
# Calculate factor scores for each observation
factor_scores = fa.transform(df[items])

# Create a DataFrame containing the factor scores
factor_scores_df = pd.DataFrame(
    factor_scores,  # Store the calculated factor scores
    columns=["Factor Value", "Factor Leader", "Factor Latest"],  # Assign names to the factor score columns
    index=df.index  # Preserve the original DataFrame index
)

# Add the brand associated with each observation
factor_scores_df["brand"] = df["brand"]

# Calculate the average factor scores for each brand
brand_factors = (
    factor_scores_df
    .groupby("brand")[["Factor Value", "Factor Leader", "Factor Latest"]]  # Group by brand and select the factor score columns
    .mean()  # Calculate the mean factor scores for each brand
)

# Output
print("\nReport:")
print("-----------------------------")
print(brand_factors)